# Notebook 01 — I/O Paths and the Config System

## What you will learn
1. How the config.yaml file is structured and why it matters
2. How to set input/output directories for your assets
3. How config values flow through the codebase
4. How to override specific settings without editing config.yaml
5. How to generate a run summary report

---

## The config.yaml philosophy

Every simulation parameter lives in `config.yaml`.  No hardcoded paths or
magic numbers in source code.  This means:

- **Reproducibility:** share the YAML and anyone can reproduce your run
- **Discoverability:** all tuneable parameters in one place
- **Safety:** changing config doesn't require touching Python source

The structure follows the simulation stages:

```yaml
problem:     # What are we solving?
mapdl:       # How to connect to MAPDL
geometry:    # Where to find the mesh
mesh:        # Quality thresholds and sizing
elements:    # Element type and KEYOPT settings
materials:   # List of material definitions
bcs:         # Boundary conditions
solver:      # Newton-Raphson and convergence settings
hfss:        # AEDT/HFSS configuration (for EM)
resources:   # CPU/RAM/port settings
output:      # Where to write results
```

In [ ]:
import sys; sys.path.insert(0, '..')
import yaml
from pathlib import Path

# ── Load and explore the config ────────────────────────────────────────────
config_path = Path('..') / 'config.yaml'
with open(config_path, encoding='utf-8') as fh:
    cfg = yaml.safe_load(fh)

print('Top-level config keys:')
for key, val in cfg.items():
    vtype = type(val).__name__
    preview = str(val)[:60].replace('\n','') if not isinstance(val, (dict, list)) else f'({vtype})'
    print(f'  {key:<16}  {vtype:<10}  {preview}')

In [ ]:
# ── Set and verify I/O paths ───────────────────────────────────────────────
from pathlib import Path

# Input paths (where to find geometry, materials data, etc.)
geometry_source = cfg['geometry']['source']
print(f'Geometry source: {geometry_source}')

if geometry_source == 'cdb':
    cdb_path = Path(cfg['geometry'].get('cdb_path', ''))
    if cdb_path.exists():
        print(f'  ✓ CDB file found: {cdb_path} ({cdb_path.stat().st_size/1024:.1f} KB)')
    else:
        print(f'  ✗ CDB file NOT found: {cdb_path}')
        print('    Update geometry.cdb_path in config.yaml')
elif geometry_source == 'parametric':
    print('  → Parametric: no input file needed')

# Output directory
out_dir = Path(cfg['output']['dir'])
out_dir.mkdir(parents=True, exist_ok=True)
print(f'\nOutput directory: {out_dir.resolve()}')
print(f'  export_csv = {cfg["output"].get("export_csv", True)}')
print(f'  export_vtk = {cfg["output"].get("export_vtk", True)}')

# MAPDL working directory
run_loc = cfg['mapdl'].get('run_location')
print(f'\nMAPDL working dir: {run_loc or "(OS temp dir)"}')

In [ ]:
# ── Override config values without editing the YAML ───────────────────────
# Use dict.update() or a deep merge for testing different parameters

from ams.multiphysics.pipeline import _deep_merge

# Run with a finer mesh for a one-off check
override = {
    'mesh': {'global_esize_m': 0.002},  # finer mesh
    'solver': {'nlgeom': True},          # enable large deformation
    'output': {'dir': '../outputs/fine_mesh_run'},
}

cfg_modified = _deep_merge(cfg, override)

print('Modified config (changes only):')
print(f'  mesh.global_esize_m = {cfg_modified["mesh"]["global_esize_m"]}')
print(f'  solver.nlgeom       = {cfg_modified["solver"]["nlgeom"]}')
print(f'  output.dir          = {cfg_modified["output"]["dir"]}')
print(f'\nOriginal untouched:')
print(f'  mesh.global_esize_m = {cfg["mesh"]["global_esize_m"]}')

In [ ]:
# ── Generate a run summary ─────────────────────────────────────────────────
import json
from datetime import datetime

summary = {
    'timestamp':     datetime.now().isoformat(),
    'problem_name':  cfg['problem']['name'],
    'physics':       cfg['problem']['physics'],
    'geometry':      cfg['geometry']['source'],
    'element_type':  cfg['elements']['type'],
    'solver_type':   cfg['solver']['type'],
    'nlgeom':        cfg['solver']['nlgeom'],
    'output_dir':    str(Path(cfg['output']['dir']).resolve()),
    'mapdl_port':    cfg['mapdl']['port'],
}

print('Run Summary:')
print(json.dumps(summary, indent=2))

# Save to output directory
summary_path = Path(cfg['output']['dir']) / 'run_summary.json'
Path(cfg['output']['dir']).mkdir(parents=True, exist_ok=True)
with open(summary_path, 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, indent=2)
print(f'\nSummary saved → {summary_path}')

---

## Summary

| Task | Code |
|------|------|
| Load config | `yaml.safe_load(open('config.yaml'))` |
| Override values | `_deep_merge(cfg, {'key': value})` |
| Create output dir | `Path(cfg['output']['dir']).mkdir(parents=True, exist_ok=True)` |
| Check input file | `Path(cfg['geometry']['cdb_path']).exists()` |

**Next:** `02_resource_management.ipynb`